In [1]:
import os
import json
import traceback
import pandas as pd
from dotenv import load_dotenv

In [2]:
load_dotenv() #it load .env files stuff in my local env

True

In [3]:
# import langchain
# from langchain.chat_models import ChatOpenAI

from langchain.llms import HuggingFacePipeline
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from transformers import BitsAndBytesConfig
from transformers import GPT2Tokenizer, GPT2Model, GPT2LMHeadModel

/Users/daniyalkhan/Documents/AI/Projects/mcq_generator/mcq_gen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
key= os.getenv("HF_API_KEY")

In [5]:
model_id = "google/flan-t5-large"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_id,
                                          use_auth_token= key)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

hf_pipeline = pipeline(
    "text2text-generation",  
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    #device= 'mps'  
)


/Users/daniyalkhan/Documents/AI/Projects/mcq_generator/mcq_gen/lib/python3.10/site-packages/transformers/models/auto/tokenization_auto.py:786: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/Users/daniyalkhan/Documents/AI/Projects/mcq_generator/mcq_gen/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [7]:
#llm= ChatOpenAI= (openai_api_key=KEY, model_name-"gpt-3.5-turbo", temperature=0.5)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

/var/folders/c2/n7zjlrp926z3_2df0tcnnf1h0000gn/T/ipykernel_2987/2260276751.py:2: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [8]:
prompt = "write this properly with proper grammar- mine name is daniyal"
result = llm(prompt)

print(result)

/var/folders/c2/n7zjlrp926z3_2df0tcnnf1h0000gn/T/ipykernel_2987/1870510362.py:2: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = llm(prompt)


mine name is daniyal


In [9]:
llm

HuggingFacePipeline(pipeline=<transformers.pipelines.text2text_generation.Text2TextGenerationPipeline object at 0x105bbca90>)

In [10]:
from langchain.llms import OpenAI 
from langchain.prompts import PromptTemplate 
from langchain.chains import LLMChain 
from langchain.chains import SequentialChain 
from langchain.callbacks import get_openai_callback 
import PyPDF2

In [14]:
RESPONSE_JSON = {
"1": {
"mcq": "multiple choice question",
"options": {
"a": "choice here",
"b": "choice here",
"c": "choice here",
"d": "choice here"
}, 
"correct": "correct answer"
},
"2": {
"mcq": "multiple choice question",
"options": {
"a": "choice here",
"b": "choice here",
"c": "choice here",
"d": "choice here",
},
"correct": "correct answer"
}
}

In [11]:
TEMPLATE="""
Text: {text}
You are an expert MCQ maker. Given the above text, it is your job to \ 
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide.\
Ensure to make {number} MCQs
### RESPONSE JSON 
{response_json}
"""

In [15]:
quiz_generation_prompt = PromptTemplate(
    input_variables= ["text", "number", "subject" , "tone", "response_json"],
    template=TEMPLATE
)

In [16]:
quiz_chain=LLMChain(llm= llm, prompt= quiz_generation_prompt, output_key="quiz", verbose=True)

/var/folders/c2/n7zjlrp926z3_2df0tcnnf1h0000gn/T/ipykernel_2987/1179640949.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  quiz_chain=LLMChain(llm= llm, prompt= quiz_generation_prompt, output_key="quiz", verbose=True)


In [17]:
TEMPLATE="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [18]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE)

In [20]:
review_chain=LLMChain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=True)

In [21]:
#now we create sequential chain for connecting both chains

generate_evaluate_chain=SequentialChain(chains=[quiz_chain, review_chain], input_variables=["text", "number", "subject", "tone", "response_json"],
                                        output_variables=["quiz", "review"], verbose=True,)

In [22]:
file_path= "/Users/daniyalkhan/Documents/AI/Projects/mcq_generator/data.txt"
with open(file_path, 'r') as file:
    TEXT = file.read()

In [23]:
 # Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [24]:
NUMBER=5 
SUBJECT="biology"
TONE="simple"

In [25]:
#https://python.langchain.com/docs/modules/model_io/llms/token_usage_tracking

#How to setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )

/var/folders/c2/n7zjlrp926z3_2df0tcnnf1h0000gn/T/ipykernel_2987/2654919708.py:5: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response=generate_evaluate_chain(
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Token indices sequence length is longer than the specified maximum sequence length for this model (649 > 512). Running this sequence through the model will result in indexing errors


Prompt after formatting:

Text: Biology is the scientific study of life.[1][2][3] It is a natural science with a broad scope but has several unifying themes that tie it together as a single, coherent field.[1][2][3] For instance, all organisms are made up of cells that process hereditary information encoded in genes, which can be transmitted to future generations. Another major theme is evolution, which explains the unity and diversity of life.[1][2][3] Energy processing is also important to life as it allows organisms to move, grow, and reproduce.[1][2][3] Finally, all organisms are able to regulate their own internal environments.[1][2][3][4][5]

Biologists are able to study life at multiple levels of organization,[1] from the molecular biology of a cell to the anatomy and physiology of plants and animals, and evolution of populations.[1][6] Hence, there are multiple subdisciplines within biology, each defined by the nature of their research questions and the tools that they use.[7][

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.
Prompt after formatting:

You are an expert english grammarian and writer. Given a Multiple Choice Quiz for biology students.You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
"1": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "2": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "3": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "4": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "ch

In [26]:
response

{'text': 'Biology is the scientific study of life.[1][2][3] It is a natural science with a broad scope but has several unifying themes that tie it together as a single, coherent field.[1][2][3] For instance, all organisms are made up of cells that process hereditary information encoded in genes, which can be transmitted to future generations. Another major theme is evolution, which explains the unity and diversity of life.[1][2][3] Energy processing is also important to life as it allows organisms to move, grow, and reproduce.[1][2][3] Finally, all organisms are able to regulate their own internal environments.[1][2][3][4][5]\n\nBiologists are able to study life at multiple levels of organization,[1] from the molecular biology of a cell to the anatomy and physiology of plants and animals, and evolution of populations.[1][6] Hence, there are multiple subdisciplines within biology, each defined by the nature of their research questions and the tools that they use.[7][8][9] Like other sci

In [27]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

Total Tokens:0
Prompt Tokens:0
Completion Tokens:0
Total Cost:0.0


In [29]:
quiz=response.get("quiz")

In [42]:
# quiz=json.loads(quiz)

In [43]:
#unable to do as we are not getting dict in return instead getting string, as not using openai api

In [40]:
quiz[:-1]

'"1": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "2": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "3": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "4": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "5": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "6": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "7": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here", "8": "mcq": "multiple choice question", "options": "a": "choice here", "b": "choic

In [44]:
# quiz_table_data = []
# for key, value in quiz.items():
#     mcq = value["mcq"]
#     options = " | ".join(
#         [
#             f"{option}: {option_value}"
#             for option, option_value in value["options"].items()
#             ]
#         )
#     correct = value["correct"]
#     quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [45]:
# quiz=pd.DataFrame(quiz_table_data)
# quiz.to_csv("machinelearning.csv",index=False)